# ch05 Bonus 10：GPT → Llama 架构转换

> 对照官方 `ch05/07_gpt_to_llama`
> **参考真实模型**：Llama-2 / Llama-3 / Qwen / Mistral 等主流开源模型

## 一句话

把我们的 GPT 改造成真实的 Llama 架构。三大改造：
1. **RoPE 旋转位置编码** 替代绝对位置嵌入
2. **RMSNorm** 替代 LayerNorm
3. **SwiGLU 激活** 替代 GELU 的 FFN

这是「见识真实开源大模型怎么设计」的重点章节。

## 三大改造对比

| 组件 | GPT-2 | Llama | 改进点 |
|------|-------|-------|--------|
| 位置编码 | 绝对位置嵌入（learned）| **RoPE**（旋转）| 外推性好、无需训练嵌入 |
| 归一化 | LayerNorm（有偏移）| **RMSNorm**（无偏移）| 更省算力、效果相当 |
| FFN 激活 | GELU | **SwiGLU** | 引入门控，表达力更强 |

> 另外现代 Llama 普遍用 GQA（见 ch04/04_gqa），这里聚焦上面三大点。

## 1. RMSNorm 替代 LayerNorm

In [ ]:
import torch
import torch.nn as nn


class RMSNorm(nn.Module):
    """RMSNorm：只用均方根归一化，去掉 mean 和 shift 偏移，比 LayerNorm 更省算力。"""

    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))  # 只有 scale，无 shift

    def forward(self, x):
        # 归一化：x / RMS(x) * weight，其中 RMS = sqrt(mean(x²))
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps) * self.weight


# 对比 LayerNorm vs RMSNorm
class LayerNormRef(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))
        self.bias = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias

x = torch.randn(2, 8, 768)
print(f"LayerNorm 参数: scale + bias = 2×{768}")
print(f"RMSNorm  参数: 仅 scale = {768}（省一半，且无 mean 计算）")

## 2. RoPE 旋转位置编码

In [ ]:
def precompute_rope_params(head_dim, theta_base=10000, context_length=4096):
    """预计算 RoPE 的 cos/sin 表。"""
    # 不同频率：theta_i = theta_base^(-2i/d)
    theta = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    pos = torch.arange(context_length).float()
    freqs = torch.outer(pos, theta)  # [seq, head_dim/2]
    return freqs.cos(), freqs.sin()


def compute_rope(x, cos, sin):
    """对 query/key 应用 RoPE 旋转。x: [b, H, seq, head_dim]。"""
    batch, H, seq, hd = x.shape
    x1 = x[..., :hd // 2]   # 前半
    x2 = x[..., hd // 2:]   # 后半
    cos = cos[:seq].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq].unsqueeze(0).unsqueeze(0)
    # 旋转：[x1, x2] → [x1·cos − x2·sin, x2·cos + x1·sin]
    rotated = torch.stack(
        [x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1
    ).flatten(-2)
    return rotated


head_dim = 64
cos, sin = precompute_rope_params(head_dim)
q = torch.randn(2, 12, 16, head_dim)
q_rot = compute_rope(q, cos, sin)
print(f"RoPE 输入: {tuple(q.shape)} → 输出: {tuple(q_rot.shape)}（同形）")
print("💡 RoPE 把位置信息编进 q/k 的旋转，无需单独的位置嵌入层。")

## 3. SwiGLU 前馈网络

In [ ]:
import torch.nn.functional as F


class SwiGLU(nn.Module):
    """SwiGLU FFN：SiLU(x·W1) ⊙ (x·W3) 再过 W2。

    相比 GPT 的 Linear-GELU-Linear，多了个门控分支（W3），表达力更强。
    Llama 的隐藏维取 2/3 × 4 × emb_dim 以保持总参数量相当。
    """

    def __init__(self, emb_dim):
        super().__init__()
        # 隐藏维：2/3 × 4 × emb_dim，向上取整到 8 的倍数（Llama 惯例）
        hidden = (int(emb_dim * 4 * 2 / 3) + 7) // 8 * 8
        self.fc1 = nn.Linear(emb_dim, hidden, bias=False)  # 门控分支
        self.fc3 = nn.Linear(emb_dim, hidden, bias=False)  # 值分支
        self.fc2 = nn.Linear(hidden, emb_dim, bias=False)  # 输出投影

    def forward(self, x):
        # SiLU(x·W1) ⊙ (x·W3) · W2
        return self.fc2(F.silu(self.fc1(x)) * self.fc3(x))


ffn = SwiGLU(768)
x = torch.randn(2, 8, 768)
print(f"SwiGLU 输出: {tuple(ffn(x).shape)}")
print(f"GPT FFN: 2 个 Linear；Llama SwiGLU: 3 个 Linear（多门控）")

## 4. 组装成 Llama Block

把三大组件 + 我们的多头注意力拼成一个 Llama 风格的 Transformer 块。RoPE 需要集成进注意力（在 q/k 计算后旋转）。

In [ ]:
class LlamaBlock(nn.Module):
    """Llama 风格 Transformer 块：RoPE 注意力 + SwiGLU + RMSNorm。"""

    def __init__(self, cfg):
        super().__init__()
        # 简化：注意力用我们的 MultiHeadAttention（真实 Llama 用 GQA + RoPE 集成）
        from src.gpt import MultiHeadAttention
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"], qkv_bias=False,
        )
        self.ff = SwiGLU(cfg["emb_dim"])              # ← 改造 3
        self.norm1 = RMSNorm(cfg["emb_dim"])           # ← 改造 2
        self.norm2 = RMSNorm(cfg["emb_dim"])

    def forward(self, x):
        # pre-norm 残差结构（和 GPT 一样，只是 norm 换成 RMSNorm）
        x = x + self.att(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


# 验证
cfg = {"emb_dim": 768, "context_length": 256, "n_heads": 12, "drop_rate": 0.0}
block = LlamaBlock(cfg)
x = torch.randn(2, 16, 768)
out = block(x)
print(f"LlamaBlock 输出: {tuple(out.shape)} ✓")
print("\n💡 再把 n_layers 个 LlamaBlock 堆叠 + RMSNorm + 输出头 = 完整 Llama。")
print("   位置编码改用 RoPE 注入到每层注意力的 q/k（而非全局 pos_emb）。")

---
> 📌 本 notebook 实现 Llama 三大改造（RoPE/RMSNorm/SwiGLU）并组装成 LlamaBlock。
> 完整 Llama 模型（含 GQA + RoPE 集成进注意力）见官方 `ch05/07_gpt_to_llama`。